# B) 팩터(수식) → LightGBM → Qlib TopkDropout 백테스트

목표
- `runs/<run_id>/data/price_with_formulas*.parquet`에 저장된 **수식 컬럼들을 feature**로 사용한다.
- forward return(예: t+3일 수익률)을 label로 만들고 **LightGBM**으로 예측한다.
- 예측 점수(pred_score)를 Qlib의 **TopkDropoutStrategy**로 넣어 백테스트하고, 문서 방식대로 `excess_return_*`를 `risk_analysis`로 계산한다.

주의
- 이 노트북은 모델/전략이 *Stage4(이벤트 진입/청산)*과 다르므로, 결과가 다르게 나오는 게 정상입니다.
- 목적은 "내 수식이 예측력(팩터)으로 의미가 있나?"를 확인하는 것입니다.
- Qlib 데이터가 로컬에 있어야 합니다(예: `~/.qlib/qlib_data/cn_data`).
- `lightgbm` 패키지가 필요합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import polars as pl

ROOT = Path("/home/user/fin/FinAgent_4090/workspace/kms/01_15_new_qlib")
RUNS_DIR = ROOT / "runs"

# ====== 설정: 분석할 run_id ======
RUN_ID = "20260122_141814"  # 바꿔서 사용

# ====== 설정: label horizon (days) ======
LABEL_HORIZON_DAYS = 3

# ====== 설정: 모델 학습 구간 (날짜) ======
# run/specs/data_split.json이 있으면 그걸 기본으로 사용하고, 없으면 아래 값을 사용
TRAIN_START = "2023-01-01"
TRAIN_END = "2024-12-31"
TEST_START = "2025-01-01"
TEST_END = "2025-12-31"

# ====== 설정: Qlib ======
PROVIDER_URI = str(Path("~/.qlib/qlib_data/cn_data").expanduser())
REGION = "cn"
BENCHMARK = "SH000905"

print("RUN_ID:", RUN_ID)


## 1) price_with_formulas 로드 + feature 컬럼 선택

- `specs/formula_bundle.json`이 있으면 그 안의 `formulas[].name`을 feature로 사용합니다.
- 없으면 parquet에서 OHLCV/메타 컬럼을 제외한 나머지를 feature로 가정합니다.


In [ ]:
run_dir = RUNS_DIR / RUN_ID
assert run_dir.exists(), run_dir

def pick_price_with_formulas(run_dir: Path) -> Path:
    cands = [run_dir / "data/price_with_formulas.parquet"] + sorted(run_dir.glob("data/price_with_formulas_iter_*.parquet"))
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(f"No price_with_formulas parquet found under {run_dir}/data")


price_path = pick_price_with_formulas(run_dir)
print("price_path:", price_path)

formula_bundle_path = run_dir / "specs" / "formula_bundle.json"
formula_names = None
if formula_bundle_path.exists():
    fb = json.loads(formula_bundle_path.read_text(encoding="utf-8"))
    formula_names = [f.get("name") for f in (fb.get("formulas") or []) if isinstance(f, dict) and f.get("name")]
    formula_names = [str(x) for x in formula_names if x]
    print("n formulas:", len(formula_names))

# schema sample
schema_cols = pl.read_parquet(price_path, n_rows=1).columns

base_cols = {"timestamp", "ticker", "open", "high", "low", "close", "volume", "factor"}
if formula_names:
    feature_cols = [c for c in formula_names if c in schema_cols]
else:
    feature_cols = [c for c in schema_cols if c not in base_cols]

print("n feature cols:", len(feature_cols))
feature_cols[:10]


## 2) label 생성 (forward return)

- label = `close(t+H) / close(t) - 1`
- ticker별로 shift(-H)로 만듭니다.


In [ ]:
H = int(LABEL_HORIZON_DAYS)

lf = (
    pl.scan_parquet(price_path)
    .select(["timestamp", "ticker", "close", *feature_cols])
    .with_columns(
        pl.col("timestamp").str.strptime(pl.Date, "%Y%m%d", strict=False).alias("dt")
    )
    .drop("timestamp")
    .sort(["ticker", "dt"])
    .with_columns(
        (pl.col("close").shift(-H) / pl.col("close") - 1.0).over("ticker").alias("label")
    )
)

df = lf.collect().to_pandas()
df = df.dropna(subset=["label"]).reset_index(drop=True)
df.head()


## 3) Train/Test split

- 기본은 `specs/data_split.json`을 읽습니다.


In [ ]:
data_split_path = run_dir / "specs" / "data_split.json"
if data_split_path.exists():
    ds = json.loads(data_split_path.read_text(encoding="utf-8"))
    TRAIN_START = ds.get("in_sample_start", TRAIN_START)
    TRAIN_END = ds.get("in_sample_end", TRAIN_END)
    TEST_START = ds.get("out_sample_start", TEST_START)
    TEST_END = ds.get("out_sample_end", TEST_END)

train_mask = (df["dt"] >= pd.to_datetime(TRAIN_START)) & (df["dt"] <= pd.to_datetime(TRAIN_END))
test_mask = (df["dt"] >= pd.to_datetime(TEST_START)) & (df["dt"] <= pd.to_datetime(TEST_END))

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

print("train rows:", len(train_df), "test rows:", len(test_df))
print("train dt:", train_df["dt"].min(), "→", train_df["dt"].max())
print("test  dt:", test_df["dt"].min(), "→", test_df["dt"].max())


## 4) LightGBM 학습

이 셀은 `lightgbm`이 설치돼 있어야 합니다.


In [ ]:
import lightgbm as lgb

X_train = train_df[feature_cols]
y_train = train_df["label"].astype(float)

# 간단한 validation split (시간 순서 유지)
train_df_sorted = train_df.sort_values("dt")
split = int(len(train_df_sorted) * 0.8)
trn = train_df_sorted.iloc[:split]
val = train_df_sorted.iloc[split:]

dtrain = lgb.Dataset(trn[feature_cols], label=trn["label"].astype(float))
dvalid = lgb.Dataset(val[feature_cols], label=val["label"].astype(float))

params = {
    "objective": "regression",
    "metric": "l2",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 42,
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dvalid],
    callbacks=[lgb.early_stopping(50)],
)

pred = model.predict(test_df[feature_cols])
test_df = test_df.copy()
test_df["pred"] = pred
test_df[["dt","ticker","label","pred"]].head()


## 5) Qlib TopkDropoutStrategy 백테스트

Qlib 문서 예시와 동일하게:
- `pred_score`: MultiIndex (datetime, instrument) -> score
- `TopkDropoutStrategy(signal=pred_score, topk=..., n_drop=...)`
- `backtest(...)` 후 `risk_analysis(report["return"] - report["bench"] [- report["cost"]])`


In [ ]:
import qlib
from qlib.constant import REG_CN
from qlib.backtest import backtest, executor
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.strategy import TopkDropoutStrategy

# init qlib
qlib.init(provider_uri=PROVIDER_URI, region=REG_CN if REGION == "cn" else REGION)

# pred_score: MultiIndex(datetime, instrument)
pred_score = (
    test_df[["dt","ticker","pred"]]
    .rename(columns={"dt":"datetime","ticker":"instrument"})
    .set_index(["datetime","instrument"])["pred"]
    .sort_index()
)

FREQ = "day"

STRATEGY_CONFIG = {
    "topk": 50,
    "n_drop": 5,
    "signal": pred_score,
}

EXECUTOR_CONFIG = {
    "time_per_step": "day",
    "generate_portfolio_metrics": True,
}

backtest_config = {
    "start_time": TEST_START,
    "end_time": TEST_END,
    "account": 100000000,
    "benchmark": BENCHMARK,
    "exchange_kwargs": {
        "freq": FREQ,
        "limit_threshold": 0.095,
        "deal_price": "close",
        "open_cost": 0.0005,
        "close_cost": 0.0015,
        "min_cost": 5,
    },
}

strategy_obj = TopkDropoutStrategy(**STRATEGY_CONFIG)
executor_obj = executor.SimulatorExecutor(**EXECUTOR_CONFIG)

portfolio_metric_dict, indicator_dict = backtest(executor=executor_obj, strategy=strategy_obj, **backtest_config)

analysis_freq = "1day"  # day == 1day
report_normal, positions_normal = portfolio_metric_dict.get(analysis_freq)

analysis = {}
analysis["excess_return_without_cost"] = risk_analysis(report_normal["return"] - report_normal["bench"], freq=analysis_freq)
analysis["excess_return_with_cost"] = risk_analysis(report_normal["return"] - report_normal["bench"] - report_normal["cost"], freq=analysis_freq)

analysis_df = pd.concat(analysis)
analysis_df
